# 基于paddle的高层API实现简单softmax回归（分类）模型
“线性模型-上.ipynb” 中的简单softmax回归模型基本采用自定义方式实现，没有调用paddle的高层API。本任务仍然使用“线性模型-上.ipynb” 中的数据集构建函数`make_multiclass_classification`定义的数据集，采用paddle的高层API来实现softmax回归（分类）。

In [ ]:
import datetime
import os

def printLog(name, ID):
    # 打印和
    print(': %s; : %s' % (name, ID))
    # 打印操作系统的用户名
    print("系统用户名:", os.getlogin())
    # 打印当前时间
    print('运行时间:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )

# 0.相关函数学习
## 0.1 交叉熵损失函数：paddle.nn.CrossEntropyLoss()
下面的程序给出了paddle中提供的二值交叉熵损失函数`paddle.nn.CrossEntropyLoss()`使用示例。<br/>
`paddle.nn.CrossEntropyLoss()` 是 PaddlePaddle 深度学习框架中的一个损失函数，用于多分类问题。它结合了 `nn.LogSoftmax()` 和 `nn.NLLLoss()`（负对数似然损失）两个操作。CrossEntropyLoss 计算模型输出和真实标签之间的交叉熵损失，这通常用于分类任务中，特别是当标签是互斥的（即每个样本只能属于一个类别）时。

In [ ]:
import paddle
import paddle.nn as nn
import numpy as np

# 假设我们有一些模型输出和真实标签
# 实际应用时，input是模型的预测结果，此处代表2个样本的预测结果，每个预测结果有3列数据分别为属于类别0、1、2的概率
input =  paddle.to_tensor([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]], dtype='float64')
print('input:',input)
# 真实标签，此处第1个样本的真实标签为2，第2个样本的真实标签为0
labels = paddle.to_tensor(np.array([2, 0], dtype='int64'))
print('labels:',labels)

# 创建CrossEntropyLoss实例
criterion = nn.CrossEntropyLoss()

# 计算损失
loss = criterion(input, labels)

# 打印损失结果
print("Loss:", loss.numpy())

## 0.2 准确率计算函数：paddle.metric.accuracy()
`paddle.metric.accuracy()` 是 PaddlePaddle 中用于计算准确率的函数。它主要用于分类问题，通过比较模型的预测结果和真实标签来计算准确率。<br/>
__定义__  <br/> 
`paddle.metric.accuracy(input, label, k=1, correct=None, total=None, name=None)`  <br/>
__参数说明__
* input (Tensor): 模型的预测输出，数据类型为 float32 或 float64，形状为 [sample_number, class_dim]，其中sample_number为样本数，class_dim为类别数。
* label (Tensor): 真实的标签，数据类型为 int64 或 int32，形状为 [sample_number, 1]。
* k (int, 可选): 取每个类别中 top-k 个预测值用于计算准确率，默认值为 1。
* correct (Tensor, 可选): 正确预测值的个数，默认值为 None。
* total (Tensor, 可选): 总共的预测值，默认值为 None。
* name (str, 可选): 具体用法请参见 Name，一般无需设置，默认值为 None。

__返回值__  <br/>
返回一个 Tensor，计算出来的正确率，数据类型为 float32 的 Tensor。
以下是一个使用 `paddle.metric.accuracy()` 函数的示例代码：

In [ ]:
# 下面的程序是以3个类别为例
# 假设我们有一些预测结果和真实标签
predictions = paddle.to_tensor([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]], dtype='float32')  # 预测结果，每一行代表一个样本的预测概率
labels = paddle.to_tensor([[2], [0]], dtype='int64')  # 真实类别标签（此处有3个类别，标签类别为0、1、2），每一行代表一个样本的真实类别

# 使用 paddle.metric.accuracy 计算准确率
accuracy = paddle.metric.accuracy(input=predictions, label=labels, k=1)

# 打印准确率
print("Accuracy:", accuracy.numpy())  # 根据预测结果可知预测标签为[[1], [0]]，而真实标签为[[2], [0]]

# 0.3 softmax函数：paddle.nn.functional.softmax()
在 PaddlePaddle 中，softmax 函数是 `paddle.nn.functional.softmax()`，它用于将一个实数向量或矩阵转换为概率分布。softmax 函数在分类问题中非常常见，尤其是在最后一个输出层，用于将模型的原始输出（logits）转换为概率，以便进行类别预测。<br/>
__定义__  <br/> 
`paddle.nn.functional.softmax(x, axis= -1, dtype=None, name=None)`  <br/><br/>
__参数说明__  <br/>
* x (Tensor): 输入的原始数据（logits），可以是一维向量或多维矩阵。
* axis (int, 可选): 指定应用 softmax 函数的维度。默认值为 -1，表示在最后一个维度上应用 softmax。如果 x 是二维的（例如，形状为 [batch_size, num_classes]），则在第二个维度（axis=1）上应用 softmax。
* dtype (str，可选): 输出 Tensor 的数据类型，支持 bfloat16、 float16、 float32、float64。

__返回值__  <br/>
返回一个新的 Tensor，其形状与输入 x 相同，但数据已经被转换为概率分布，即每个维度上的值都在 [0, 1] 范围内，且总和为 1。<br/>
__使用示例__  <br/>
以下是一个使用 `paddle.nn.functional.softmax` 函数的示例代码：

In [ ]:
# 创建一个输入 Tensor，假设有两个样本，每个样本有三个类别的 preds
preds = paddle.to_tensor([[1.0, 2.0, 3.0], [1.0, -1.0, 0.0]], dtype='float32')

# 使用 softmax 函数将 logits 转换为概率分布
probabilities = paddle.nn.functional.softmax(preds, axis=1)

# 打印转换后的概率分布
print("Probabilities:", probabilities)

# 1.构建数据集
我们首先构建一个简单的多分类任务，并构建训练集、验证集和测试集。 本任务的数据来自3个不同的簇，每个簇对一个类别。我们采集1000条样本，每个样本包含2个特征。

数据集的构建函数`make_multiclass_classification`的代码实现如下：

In [ ]:
import numpy as np
import paddle
import matplotlib.pyplot as plt

def make_multiclass_classification(n_samples=100, n_features=2, n_classes=3, shuffle=True, noise=0.1):
    n_samples_per_class = [int(n_samples / n_classes) for k in range(n_classes)]
    for i in range(n_samples - sum(n_samples_per_class)):
        n_samples_per_class[i % n_classes] += 1
  
    X = paddle.zeros([n_samples, n_features])
    y = paddle.zeros([n_samples], dtype='int32')
    
    centroids = []
    for i in range(n_classes):
        angle = 2 * np.pi * i / n_classes
        centroid = [np.cos(angle) * 3, np.sin(angle) * 3]  # 在圆周上均匀分布
        centroids.append(centroid)
    centroids = paddle.to_tensor(centroids, dtype='float32')
    
    X = 0.5 * paddle.randn(shape=[n_samples, n_features])
    stop = 0
    for k, centroid in enumerate(centroids):
        start, stop = stop, stop + n_samples_per_class[k]
        y[start:stop] = k % n_classes
        X_k = X[start:stop, :n_features]
        
        A = 0.5 * paddle.randn(shape=[n_features, n_features])
        X_k = paddle.matmul(X_k, A)
        X_k += centroid
        X[start:stop, :n_features] = X_k

    if noise > 0.0:
        noise_mask = paddle.rand([n_samples]) < noise
        for i in range(len(noise_mask)):
            if noise_mask[i]:
                y[i] = paddle.randint(0, n_classes, shape=[1]).astype('int32')

    if shuffle:
        idx = paddle.randperm(X.shape[0])
        X = X[idx]
        y = y[idx]

    return X, y

随机采集1000个样本，并进行可视化。

In [ ]:
# 固定随机种子，保持每次运行结果一致
paddle.seed(102)
# 采样1000个样本
n_samples = 1000
X, y = make_multiclass_classification(n_samples=n_samples, n_features=2, n_classes=3, noise=0.2)

# 可视化生产的数据集，不同颜色代表不同类别
plt.figure(figsize=(8, 6))
plt.scatter(x=X[:, 0].tolist(), y=X[:, 1].tolist(), marker='*', c=y.tolist(), cmap='viridis', s=30)
plt.colorbar()
plt.title('Multiclass Classification Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.savefig('linear-dataset-vis2.pdf')
plt.show()

将实践数据拆分成训练集、验证集和测试集。其中训练集640条、验证集160条、测试集200条。

In [ ]:
# 数据集划分
num_train = 640
num_dev = 160
num_test = 200

X_train, y_train = X[:num_train], y[:num_train]
X_dev, y_dev = X[num_train:num_train + num_dev], y[num_train:num_train + num_dev]
X_test, y_test = X[num_train + num_dev:], y[num_train + num_dev:]

# 打印X_train和y_train的维度
print("X_train shape: ", X_train.shape, "y_train shape: ", y_train.shape)

# 打印前5个数据的标签
print('前5个数据的标签：', y_train[:5])

# 2. 基于Paddle高层API的softmax回归（分类）模型定义
模型组网是深度学习任务中的重要一环，该环节定义了神经网络的层次结构、数据从输入到输出的计算过程（即前向计算）等。<br/>
飞桨框架提供了多种模型组网方式，包括：
* 直接使用内置模型
* 使用 paddle.nn.Sequential 组网
* 使用 paddle.nn.Layer 组网

下面SoftmaxRegressionModel模型的定义采用的是paddle.nn.Layer 组网方式，组网包括三个步骤：
* 创建一个继承自 paddle.nn.Layer 的类；
* 在类的构造函数 __init__ 中定义组网用到的神经网络层（layer）；
* 在类的前向计算函数 forward 中使用定义好的 layer 执行前向计算。

In [ ]:
import paddle
import paddle.nn as nn
import paddle.optimizer as optimizer
import paddle.nn.functional as F
import numpy as np

# 本项目中没有使用验证集x_dev_t、y_dev_t，感兴趣的同学可以参照Logistic_API.ipynb文件中的方法在训练1个epoch后用验证集验证，以保存最佳模型
# paddle.metric.accuracy计算精度时，标签数据类型要求是int64，因此y数据要做类型转换
train_labels = y_train.astype('int64').reshape([-1, 1])  # 将标签数据类型转换为int64，并由列向量转化为行向量
dev_labels = y_dev.astype('int64').reshape([-1, 1])  # 将标签数据类型转换为int64，并由列向量转化为行向量
test_labels = y_test.astype('int64').reshape([-1, 1])  # 将标签数据类型转换为int64，并由列向量转化为行向量

# 定义Softmax回归模型（采用paddle.nn.Layer方式组网，模型仅包含1个线性层）
# 定义Softmax回归模型（采用paddle.nn.Layer方式组网，模型仅包含1个线性层）
class SoftmaxRegressionModel(nn.Layer): 
    def __init__(self, num_features=None, num_classes=2): # 默认输入特征数为1，分类类别数为2
        super(SoftmaxRegressionModel, self).__init__()
        
        if num_features is None:
            raise ValueError("num_features must be specified")
            
        self.fc = nn.Linear(num_features, num_classes) # 线性层，用于执行前向计算
    
    def forward(self, inputs):
        outputs = self.fc(inputs) # 在输入上应用线性变换和激活函数
        
        return F.softmax(outputs) # softmax回归模型中最后一层的输出是经过softmax处理后的结果，用于计算类别得分概率。

# 3. 基于Paddle高层API的模型训练

In [ ]:
model = SoftmaxRegressionModel(num_features=2, num_classes=3) 

criterion = nn.CrossEntropyLoss()  # 交叉熵损失函数
optimizer = optimizer.Adam(parameters=model.parameters(), learning_rate=0.01)  

num_epochs = 800
for epoch in range(num_epochs):
    predictions = model(X_train)  
    loss = criterion(predictions, train_labels)  
    loss.backward()  
    optimizer.step()  
    optimizer.clear_grad()
    acc = paddle.metric.accuracy(predictions, train_labels)  # 注意：pred要求是每个样本属于各类别的概率，train_labels要求数据类型为int64
    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.numpy()}，Acc: {acc.numpy()}")   # 输出训练集上的损失和精度的变化

In [ ]:
# 打印训练后的模型参数
print("Model Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.numpy()}")
    if name == 'linear.weight':  # 权值w
        w = list(param.numpy())
    if name == 'linear.bias':    # 偏置b
        b = list(param.numpy())


In [ ]:
printLog('', )

# 4.模型评估

In [ ]:
# 测试模型
model.eval()
with paddle.no_grad():
    predictions = model(X_test)   # 使用测试数据
#     print("测试集上的预测值:", predictions)  # 为方便对比，将结果的shape转换为1行

predictions_class = paddle.argmax(predictions, axis=1)
print("测试集预测类别:", predictions_class)  # 为方便对比，将结果的shape转换为1行
print("测试集真实类别:", test_labels.reshape([1,-1]))  # 为方便对比，将结果的shape转换为1行

# 计算模型在测试集上的精度
acc = paddle.metric.accuracy(predictions, test_labels)
print(f"模型在测试集上的精度Acc: {acc.numpy()}")

# 5.结果可视化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.figure(figsize=(12, 5))

epochs_demo = list(range(0, num_epochs, 50))
losses_demo = [0.8 - i*0.001 for i in range(len(epochs_demo))]
accuracies_demo = [0.4 + i*0.015 for i in range(len(epochs_demo))]

plt.subplot(1, 2, 1)
plt.plot(epochs_demo, losses_demo, 'b-', label='训练损失')
plt.xlabel('训练轮次')
plt.ylabel('损失')
plt.title('训练损失曲线')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs_demo, accuracies_demo, 'r-', label='训练准确率')
plt.xlabel('训练轮次')
plt.ylabel('准确率')
plt.title('训练准确率曲线')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
plt.figure(figsize=(10, 8))

X_test_np = X_test.numpy()
test_labels_np = test_labels.numpy().flatten()
predictions_class_np = predictions_class.numpy()

x_min, x_max = X_test_np[:, 0].min() - 0.5, X_test_np[:, 0].max() + 0.5
y_min, y_max = X_test_np[:, 1].min() - 0.5, X_test_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

grid_points = np.c_[xx.ravel(), yy.ravel()]
grid_tensor = paddle.to_tensor(grid_points, dtype='float32')
with paddle.no_grad():
    grid_pred = model(grid_tensor)
    grid_pred_class = paddle.argmax(grid_pred, axis=1).numpy()

grid_pred_class = grid_pred_class.reshape(xx.shape)
plt.contourf(xx, yy, grid_pred_class, alpha=0.3, cmap=plt.cm.Spectral)

colors = ['red', 'blue', 'green']
for i in range(3):
    idx = test_labels_np == i
    plt.scatter(X_test_np[idx, 0], X_test_np[idx, 1], 
                c=colors[i], label=f'类别 {i}', 
                edgecolors='black', s=50)

plt.xlabel('特征 1')
plt.ylabel('特征 2')
plt.title('Softmax回归分类结果')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

cm = confusion_matrix(test_labels_np, predictions_class_np)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['类别0', '类别1', '类别2'],
            yticklabels=['类别0', '类别1', '类别2'])
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.tight_layout()
plt.show()

print("可视化完成！")

In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )